# importing KML file and sorting and managing data 

In [ ]:
import os
import geopandas as gpd 
import pandas as pd
import shapely.wkt as wkt

from pyautocad import Autocad, APoint
import pandas as pd
from pyproj import Transformer

In [ ]:
GI_Data = os.path.abspath("inputs/BH_Data.kml")
BH2_path = os.path.abspath("exports/GI Locations.kml")

GI_df = gpd.read_file(GI_Data, driver="KML")   # driver name is "KML"

print("Number of rows & columns in GeoDataFrame:",GI_df.shape)
print("Column Names:", GI_df.columns)
print(GI_df.info())
print(GI_df.isnull().sum())

In [ ]:
orig_cols = GI_df.columns
GI_df.columns = GI_df.columns.str.strip().str.replace('\n',' ').str.lower()

keep = ['name','geometry']
keep_existing = [c for c in keep if c in GI_df.columns]
GI_df = GI_df[keep_existing].copy()

GI_df.rename(columns={'name': 'Name', 'geometry': 'geometry'}, inplace=True)
GI_df


In [ ]:
gdf = gpd.GeoDataFrame(GI_df, geometry='geometry', copy=True)


if gdf.geometry.dtype == object and isinstance(gdf.geometry.iloc[0], str):
    gdf.geometry = gdf.geometry.apply(wkt.loads)

gdf = gdf[gdf.geometry.geom_type == 'Point'].copy()
gdf['x'] = gdf.geometry.x
gdf['y'] = gdf.geometry.y

gdf['x'] = gdf['x']
gdf['y'] = gdf['y']

gdf = gdf[['Name','x','y']]

# save
# gdf.to_csv('GI_df.csv', index=False)


gdf

# Pyautocad plotting to autocad

In [ ]:
# checking if autocad connection works
from pyautocad import Autocad

print("START")
try:
    # do not create AutoCAD if not running
    cad = Autocad(create_if_not_exists=False)
    print("Autocad object:", cad)
    cad.prompt("Hello,World\n")
    print("prompt() called")
    doc = cad.ActiveDocument
    print("ActiveDocument:", bool(doc))
    try:
        ms = doc.ModelSpace
        print("ModelSpace available")
    except Exception as e:
        print("ModelSpace error:", e)
except Exception as e:
    print("Connection ERROR:", e)
print("END")


In [ ]:
# NOTE: This function currently converts lat/lon (EPSG:4326) -> British National Grid (EPSG:27700).
# If your CSV already has eastings/northings, you can skip this and use the raw x/y.
def to_transform(x_vals, y_vals):
    t = Transformer.from_crs("epsg:4326", "epsg:27700", always_xy=True)
    x_cord, y_cord = t.transform(x_vals, y_vals)
    return x_cord, y_cord

def plotting_to_cad(cadModel, ID_name, Xcord, Ycord):
    block_name = "BOREHOLE-A"   # The EXACT name of block in AutoCAD
    attribute_tag = "BH_No"     # Attribute tag to set on the block

    for i, (ID, x, y) in enumerate(zip(ID_name, Xcord, Ycord)):
        try:
            print(f"{i}. ID_Name: {ID} | x: {x} | y: {y}")
            centre = APoint(float(x), float(y), 0)
            label = APoint(float(x) + 5, float(y) + 5, 0)
            text_height = 5

            # Simple geometry
            try:
                cadModel.AddCircle(centre, 1)
            except Exception as e:
                print(f"AddCircle error: {e}")

            try:
                cadModel.AddText(str(ID), label, text_height)
            except Exception as e:
                print(f"AddText error: {e}")

            # Insert the block
            try:
                block_ref = cadModel.InsertBlock(centre, block_name, 1, 1, 1, 0.0)
                print("Inserted block:", bool(block_ref))
            except Exception as e:
                print(f"InsertBlock error: {e}")
                continue

            # Set the attribute by tag (if present)
            try:
                attrs = block_ref.GetAttributes()
                if attrs:
                    updated = False
                    for att in attrs:
                        if str(att.TagString).strip().lower() == attribute_tag.lower():
                            att.TextString = str(ID)
                            updated = True
                            break
                    if not updated:
                        print(f"Attribute '{attribute_tag}' not found on block '{block_name}'.")
                else:
                    print(f"No attributes found on the inserted block '{block_name}'.")
            except Exception as e:
                print(f"Attribute update error: {e}")

        except Exception as e:
            print(f"Row {i} error: {e}")

def main():
    csv_path = "GI_df.csv"
    if not os.path.exists(csv_path):
        print(f"CSV not found: {csv_path}")
        return

    try:
        df = pd.read_csv(csv_path)
    except Exception as e:
        print(f"Failed to read '{csv_path}': {e}")
        return

    # Validate expected columns exist
    for col in ("x", "y", "Name"):
        if col not in df.columns:
            print(f"Missing required column '{col}' in '{csv_path}'.")
            return

    # If your x/y are already eastings/northings in the drawing CRS, skip the transform:
    # X_cord, Y_cord = df["x"].values, df["y"].values
    X_cord, Y_cord = to_transform(df["x"].values, df["y"].values)
    ID_name = df["Name"].values

    try:
        cadApp = Autocad()
        cadDoc = cadApp.ActiveDocument
        cadModel = cadDoc.ModelSpace
    except Exception as e:
        print(f"Failed to connect to AutoCAD: {e}")
        return

    cadApp.prompt("Hello, World\n")
    plotting_to_cad(cadModel, ID_name, X_cord, Y_cord)
    print("Done.")

if __name__ == "__main__":
    main()
